# Forest Plot Table Extraction

Extracts tabular data from forest plot images in clinical research PDFs using Gemini.

Uses modules from `agents.forest_plot` and `shared`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# Ensure project root is on sys.path so imports work
PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared.client import client, DEFAULT_MODEL
from shared.pdf import render_pages
from shared.eval import compare_tables, print_accuracy_summary
from agents.forest_plot.extract import extract_forest_plots_from_page, stitch_forest_plot_results

print(f"Client ready, model: {DEFAULT_MODEL}")

In [ ]:
# Configuration
PDF_PATH = str(PROJECT_ROOT / "example sources" / "Effeccts of Statins - Lancet (1).pdf")
PAGES = [7]  # 1-indexed
MODEL = DEFAULT_MODEL
DPI = 300
OUTPUT_DIR = Path("test_data") / "output"
GROUND_TRUTH_CSV = Path("test_data") / "ground_truth" / "forest_plot.csv"

print(f"PDF: {PDF_PATH}")
print(f"Pages: {PAGES}")
print(f"Model: {MODEL}")

In [ ]:
# Render and display pages
page_images = render_pages(PDF_PATH, PAGES, dpi=DPI)

for page_num, img in page_images.items():
    print(f"Page {page_num}: {img.size[0]}x{img.size[1]} pixels")
    display_width = 600
    ratio = display_width / img.size[0]
    display(img.resize((display_width, int(img.size[1] * ratio))))

In [ ]:
# Extract forest plots from each page
page_results = {}
continuation_context = None

for page_num in PAGES:
    print(f"Processing page {page_num} (continuation={'yes' if continuation_context else 'no'})...")
    result = extract_forest_plots_from_page(
        client, MODEL, page_images[page_num], continuation_context=continuation_context
    )
    page_results[page_num] = result

    for i, fp in enumerate(result.forest_plots):
        print(f"  Plot {i+1}: \"{fp.title}\" — {len(fp.rows)} rows, complete={fp.plot_appears_complete}")

    if result.forest_plots and not result.forest_plots[-1].plot_appears_complete:
        last_fp = result.forest_plots[-1]
        continuation_context = {"title": last_fp.title, "headers": last_fp.headers}
    else:
        continuation_context = None

In [ ]:
# Stitch and preview results
extracted_plots = stitch_forest_plot_results(page_results)
print(f"Total logical forest plots: {len(extracted_plots)}\n")

for i, (title, df) in enumerate(extracted_plots):
    print(f"Plot {i+1}: \"{title}\" — {len(df)} rows x {len(df.columns)} columns")
    display(df)
    print()

In [ ]:
# Save to CSV
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for i, (title, df) in enumerate(extracted_plots):
    filename = OUTPUT_DIR / f"forest_plot_{i+1}.csv"
    df.to_csv(filename, index=False)
    print(f"Saved: {filename} ({len(df)} rows x {len(df.columns)} cols)")

In [ ]:
# Evaluate against ground truth
if GROUND_TRUTH_CSV.exists():
    PLOT_INDEX = 0
    df_truth = pd.read_csv(GROUND_TRUTH_CSV, dtype=str).fillna("")
    df_extracted = pd.read_csv(OUTPUT_DIR / f"forest_plot_{PLOT_INDEX+1}.csv", dtype=str).fillna("")

    print(f"Ground truth: {len(df_truth)} rows x {len(df_truth.columns)} cols")
    print(f"Extracted:    {len(df_extracted)} rows x {len(df_extracted.columns)} cols\n")

    results = compare_tables(df_truth, df_extracted)
    print_accuracy_summary(results)
else:
    print(f"Ground truth not found at {GROUND_TRUTH_CSV} — skipping eval.")